In [1]:
pip install pandas numpy scipy statsmodels arch openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# 中国商品期货 20 个 high-reliability 合约
# 波动率模型选择、传统诊断与 Hong-Lee 论文检验全流程
#
# 输入：
#   01_base_clean_panel.csv
#
# 样本：
#   20 个 high-reliability 主力连续合约
#
# 三大章节：
#   一、模型筛选
#   二、传统模型检验
#   三、论文模型检验
#
# 依赖：
#   pip install pandas numpy scipy statsmodels arch openpyxl
# ============================================================


# ============================================================
# 0. 环境与全局设置
# ============================================================

import os
import re
import json
import math
import time
import hashlib
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import pandas as pd

from scipy import stats
from scipy.optimize import minimize

from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

from arch import arch_model

warnings.filterwarnings("ignore")


# ----------------------------
# 路径设置
# ----------------------------

INPUT_CANDIDATES = [
    Path("01_base_clean_panel.csv"),
    Path("/mnt/data/01_base_clean_panel.csv"),
]

INPUT_PATH = next((p for p in INPUT_CANDIDATES if p.exists()), None)
if INPUT_PATH is None:
    raise FileNotFoundError(
        "找不到 01_base_clean_panel.csv，请确认文件在当前目录或 /mnt/data/ 下。"
    )

OUTPUT_DIR = Path("20_contract_vol_model_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input path:", INPUT_PATH)
print("Output dir:", OUTPUT_DIR.resolve())


# ----------------------------
# 核心配置
# ----------------------------

HIGH_RELIABILITY_SYMBOLS = [
    "AG", "AL", "AU",
    "CF", "CU",
    "FG", "HC", "I", "JM",
    "M", "MA", "OI", "P",
    "RB", "RU", "SN", "SR",
    "TA", "V", "Y",
]

# 是否加入 21 号候选：无条件厚尾基准模型
# False = 20 个正式主模型
# True  = 21 个模型，加入 CONSTVOL_t
INCLUDE_CONSTVOL_T = False

# 检验显著性水平
ALPHA = 0.05

# 最小样本要求
MIN_OBS_MODEL = 1000

# VaR 检验水平
VAR_LEVELS = [0.95, 0.99]

# Hong-Lee 带宽倍数
HL_BANDWIDTH_MULTIPLIERS = [0.5, 1.0, 1.5, 2.0]

# Hong-Lee 广义谱二阶导数积分设置
# W(v) 取标准正态分布，使用 Gauss-Hermite 节点近似 int g(v)dW(v)。
HL_WEIGHT_GRID_SIZE = 15
HL_WEIGHT_SCALE = 1.0
HL_RESTANDARDIZE_RESID = True
HL_SENSITIVITY_GRID_SIZES = [8, 15, 21]
HL_SENSITIVITY_WEIGHT_SCALES = [0.75, 1.00, 1.25]
RUN_HL_DATA_DRIVEN_BANDWIDTH = True
HL_DD_C_GRID = [0.75, 1.00, 1.50, 2.00, 2.50, 3.00]
HL_DD_LAMBDA_GRID = [0.25, 0.30, 0.35, 0.40]
HL_DD_MAX_RATIO = 0.20
HL_DD_PENALTY_STRENGTH = 1.0

# 多重检验调整
MULTIPLE_TEST_METHOD = "bh"  # "bh" or "holm"

# VaR 滚动回测设置
ROLLING_VAR_BACKTEST = True
VAR_ONE_STEP_RECURSIVE = True
VAR_ROLLING_INITIAL_FRACTION = 0.70
VAR_MIN_TRAIN_OBS = 750
VAR_REFIT_EVERY = 20

# Hong-Lee Monte Carlo 尺寸诊断
# 100 次适合作为 diagnostic；若要写成正式 simulation，建议提高到 300-1000。
RUN_HL_MONTE_CARLO_SIZE = True
HL_MC_REPS = 100
HL_MC_BURN = 500
HL_MC_RANDOM_SEED = 20260620
HL_MC_FAMILY_SPECS = {
    "GARCH": "03_GARCH11_t",
    "EGARCH": "08_EGARCH11_t",
    "GJR": "11_GJR11_t",
}

# arch 拟合设置
ARCH_FIT_OPTIONS = {
    "disp": "off",
    "show_warning": False,
    "options": {"maxiter": 1000},
}

# 选择主模型时优先使用 BIC
MODEL_SELECTION_CRITERION = "bic"



Input path: 01_base_clean_panel.csv
Output dir: /home/zilinm2/20_contract_vol_model_output


In [3]:
# 1. 工具函数
# ============================================================

def ensure_bool(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s
    return s.astype(str).str.lower().isin(["true", "1", "yes"])


def clean_symbol_data(df: pd.DataFrame, symbol: str) -> pd.DataFrame:
    """
    提取单个合约的可建模收益率序列。
    口径：
      ret_pct = log_ret_1d * 100
      必须 valid_return_flag == True
      必须 tradable_return_flag == True
    """
    sub = df[df["symbol"] == symbol].copy()

    if sub.empty:
        raise ValueError(f"{symbol} 数据为空。")

    sub["date"] = pd.to_datetime(sub["date"])
    sub = sub.sort_values("date").reset_index(drop=True)

    for col in ["valid_return_flag", "tradable_return_flag"]:
        if col in sub.columns:
            sub[col] = ensure_bool(sub[col])
        else:
            raise KeyError(f"缺少必要字段：{col}")

    sub = sub[
        (sub["valid_return_flag"]) &
        (sub["tradable_return_flag"]) &
        (sub["log_ret_1d"].notna())
    ].copy()

    sub["ret_pct"] = sub["log_ret_1d"].astype(float) * 100.0

    sub = sub.replace([np.inf, -np.inf], np.nan)
    sub = sub.dropna(subset=["ret_pct", "close"])

    return sub


def safe_adf(y: pd.Series) -> Dict[str, Any]:
    y = pd.Series(y).dropna()
    if len(y) < 50:
        return {"adf_stat": np.nan, "adf_pvalue": np.nan}
    try:
        res = adfuller(y, autolag="AIC")
        return {"adf_stat": res[0], "adf_pvalue": res[1]}
    except Exception:
        return {"adf_stat": np.nan, "adf_pvalue": np.nan}


def safe_ljungbox(x: pd.Series, lags: List[int]) -> Dict[str, float]:
    x = pd.Series(x).replace([np.inf, -np.inf], np.nan).dropna()
    out = {}
    for lag in lags:
        key = f"lb_p_lag{lag}"
        if len(x) <= lag + 5:
            out[key] = np.nan
            continue
        try:
            lb = acorr_ljungbox(x, lags=[lag], return_df=True)
            out[key] = float(lb["lb_pvalue"].iloc[-1])
        except Exception:
            out[key] = np.nan
    return out


def safe_arch_lm(x: pd.Series, nlags: int = 10) -> Dict[str, float]:
    x = pd.Series(x).replace([np.inf, -np.inf], np.nan).dropna()
    if len(x) <= nlags + 20:
        return {f"arch_lm_p_lag{nlags}": np.nan}
    try:
        stat, pvalue, fstat, fpvalue = het_arch(x, nlags=nlags)
        return {f"arch_lm_p_lag{nlags}": float(pvalue)}
    except Exception:
        return {f"arch_lm_p_lag{nlags}": np.nan}


def kupiec_pof_test(exceedances: np.ndarray, expected_prob: float) -> Dict[str, float]:
    """
    Kupiec Proportion of Failures test.
    exceedances: True/False array
    expected_prob: 例如 0.05 或 0.01
    """
    exc = np.asarray(exceedances).astype(bool)
    n = len(exc)
    x = int(exc.sum())

    if n == 0:
        return {
            "n": 0,
            "violations": 0,
            "violation_rate": np.nan,
            "expected_rate": expected_prob,
            "kupiec_lr": np.nan,
            "kupiec_pvalue": np.nan,
            "kupiec_pass": False,
        }

    phat = x / n

    eps = 1e-12
    p0 = min(max(expected_prob, eps), 1 - eps)
    phat_safe = min(max(phat, eps), 1 - eps)

    log_l0 = (n - x) * np.log(1 - p0) + x * np.log(p0)
    log_l1 = (n - x) * np.log(1 - phat_safe) + x * np.log(phat_safe)

    lr = -2 * (log_l0 - log_l1)
    pvalue = 1 - stats.chi2.cdf(lr, df=1)

    return {
        "n": n,
        "violations": x,
        "violation_rate": phat,
        "expected_rate": expected_prob,
        "kupiec_lr": lr,
        "kupiec_pvalue": pvalue,
        "kupiec_pass": bool(pvalue >= ALPHA),
    }


def classify_vol_state(percentile: float) -> str:
    if pd.isna(percentile):
        return "unknown"
    if percentile < 0.20:
        return "low_vol"
    if percentile < 0.80:
        return "normal_vol"
    if percentile < 0.95:
        return "high_vol"
    return "extreme_high_vol"


def reliability_by_n(n: int) -> str:
    if n >= 2000:
        return "HIGH"
    if n >= 1000:
        return "MEDIUM"
    if n >= 700:
        return "LOW"
    return "VERY_LOW"


def stable_int_seed(*items, base: int = HL_MC_RANDOM_SEED, modulo: int = 100000) -> int:
    """
    Stable deterministic seed independent of Python's randomized hash().
    """
    s = "|".join(map(str, items)).encode("utf-8")
    h = int(hashlib.md5(s).hexdigest()[:8], 16)
    return int(base + h % modulo)


def is_material_fit_warning(warning_text: str) -> bool:
    """
    Only warnings that may affect fitted model validity should downgrade model health.
    Harmless numerical/display/deprecation warnings are preserved in fit_warnings
    but do not automatically trigger fit_warning_review_needed.
    """
    if not warning_text:
        return False
    txt = str(warning_text).lower()
    material_keywords = [
        "convergence",
        "optimizer",
        "iteration",
        "failed",
        "failure",
        "incompatible",
        "invalid",
        "nonstationary",
        "non-stationary",
        "singular",
        "nan",
        "inf",
        "overflow",
        "underflow",
        "not finite",
    ]
    return any(k in txt for k in material_keywords)



In [4]:
# 2. 候选模型定义
# ============================================================

@dataclass
class ModelSpec:
    name: str
    model_type: str          # "arch" or "constvol_t"
    mean: str = "Constant"
    lags: Optional[int] = None
    vol: str = "GARCH"
    p: int = 1
    o: int = 0
    q: int = 1
    power: float = 2.0
    dist: str = "t"


def build_model_specs(include_constvol_t: bool = False) -> List[ModelSpec]:
    specs = []

    if include_constvol_t:
        specs.append(ModelSpec(
            name="00_CONSTVOL_t",
            model_type="constvol_t",
        ))

    specs.extend([
        # 01. ARCH 基准
        ModelSpec("01_ARCH5_t", "arch", mean="Constant", vol="ARCH", p=5, o=0, q=0, power=2.0, dist="t"),

        # 02-05. GARCH(1,1) 分布扩展
        ModelSpec("02_GARCH11_normal", "arch", mean="Constant", vol="GARCH", p=1, o=0, q=1, power=2.0, dist="normal"),
        ModelSpec("03_GARCH11_t",      "arch", mean="Constant", vol="GARCH", p=1, o=0, q=1, power=2.0, dist="t"),
        ModelSpec("04_GARCH11_skewt",  "arch", mean="Constant", vol="GARCH", p=1, o=0, q=1, power=2.0, dist="skewt"),
        ModelSpec("05_GARCH11_ged",    "arch", mean="Constant", vol="GARCH", p=1, o=0, q=1, power=2.0, dist="ged"),

        # 06-07. GARCH 阶数扩展
        ModelSpec("06_GARCH12_t", "arch", mean="Constant", vol="GARCH", p=1, o=0, q=2, power=2.0, dist="t"),
        ModelSpec("07_GARCH21_t", "arch", mean="Constant", vol="GARCH", p=2, o=0, q=1, power=2.0, dist="t"),

        # 08-10. EGARCH
        ModelSpec("08_EGARCH11_t",     "arch", mean="Constant", vol="EGARCH", p=1, o=1, q=1, power=2.0, dist="t"),
        ModelSpec("09_EGARCH11_skewt", "arch", mean="Constant", vol="EGARCH", p=1, o=1, q=1, power=2.0, dist="skewt"),
        ModelSpec("10_EGARCH21_t",     "arch", mean="Constant", vol="EGARCH", p=2, o=1, q=1, power=2.0, dist="t"),

        # 11-13. GJR-GARCH：方差层面非对称，power=2
        ModelSpec("11_GJR11_t",     "arch", mean="Constant", vol="GARCH", p=1, o=1, q=1, power=2.0, dist="t"),
        ModelSpec("12_GJR11_skewt", "arch", mean="Constant", vol="GARCH", p=1, o=1, q=1, power=2.0, dist="skewt"),
        ModelSpec("13_GJR21_t",     "arch", mean="Constant", vol="GARCH", p=2, o=1, q=1, power=2.0, dist="t"),

        # 14-15. TARCH / ZARCH：标准差层面非对称，power=1
        ModelSpec("14_TARCH11_t",     "arch", mean="Constant", vol="GARCH", p=1, o=1, q=1, power=1.0, dist="t"),
        ModelSpec("15_TARCH11_skewt", "arch", mean="Constant", vol="GARCH", p=1, o=1, q=1, power=1.0, dist="skewt"),

        # 16-17. APARCH：幂次可变 + 非对称
        ModelSpec("16_APARCH11_t",     "arch", mean="Constant", vol="APARCH", p=1, o=1, q=1, power=2.0, dist="t"),
        ModelSpec("17_APARCH11_skewt", "arch", mean="Constant", vol="APARCH", p=1, o=1, q=1, power=2.0, dist="skewt"),

        # 18-20. 均值项扩展
        ModelSpec("18_AR1_GARCH11_t", "arch", mean="AR", lags=1, vol="GARCH", p=1, o=0, q=1, power=2.0, dist="t"),
        ModelSpec("19_AR1_EGARCH11_t", "arch", mean="AR", lags=1, vol="EGARCH", p=1, o=1, q=1, power=2.0, dist="t"),
        ModelSpec("20_AR1_GJR11_t", "arch", mean="AR", lags=1, vol="GARCH", p=1, o=1, q=1, power=2.0, dist="t"),
    ])

    return specs


MODEL_SPECS = build_model_specs(INCLUDE_CONSTVOL_T)

print("Candidate model count:", len(MODEL_SPECS))
print([s.name for s in MODEL_SPECS])



Candidate model count: 20
['01_ARCH5_t', '02_GARCH11_normal', '03_GARCH11_t', '04_GARCH11_skewt', '05_GARCH11_ged', '06_GARCH12_t', '07_GARCH21_t', '08_EGARCH11_t', '09_EGARCH11_skewt', '10_EGARCH21_t', '11_GJR11_t', '12_GJR11_skewt', '13_GJR21_t', '14_TARCH11_t', '15_TARCH11_skewt', '16_APARCH11_t', '17_APARCH11_skewt', '18_AR1_GARCH11_t', '19_AR1_EGARCH11_t', '20_AR1_GJR11_t']


In [5]:
# 3. 模型拟合工具
# ============================================================

def params_to_json(params: Any) -> str:
    try:
        if isinstance(params, pd.Series):
            d = {str(k): float(v) for k, v in params.items()}
        elif isinstance(params, dict):
            d = {str(k): float(v) for k, v in params.items()}
        else:
            d = {}
        return json.dumps(d, ensure_ascii=False)
    except Exception:
        return "{}"


def _sum_params_by_prefix(params: Dict[str, float], prefixes: Tuple[str, ...]) -> float:
    total = 0.0
    for k, v in params.items():
        kk = str(k).lower()
        if any(kk.startswith(prefix) for prefix in prefixes):
            total += float(v)
    return total


def check_params_ok(spec: ModelSpec, params: Dict[str, float]) -> Tuple[bool, str]:
    """
    参数合理性粗筛。
    主要避免明显异常：
      t / skewt 自由度过低；
      skewt 偏度参数越界；
      GED shape 异常；
      APARCH delta 异常；
      普通 GARCH / GJR-GARCH 持久性过高或近单位根。

    注意：
      t / skew-t 自由度 <= 4 只影响 Hong-Lee 渐近推断的四阶矩条件，
      不应直接判定 GARCH / VaR 模型不可用。
    """
    try:
        for k, v in params.items():
            if not np.isfinite(v):
                return False, f"non_finite_param:{k}"

        dist = spec.dist.lower()

        # Student-t / skew-t
        if dist in ["t", "studentst"]:
            nu = params.get("nu", None)
            if nu is not None and not (2.05 < nu < 200):
                return False, "bad_t_nu"
            # nu <= 4 indicates a fourth-moment warning for Hong-Lee inference,
            # but the volatility model itself can still be useful for AIC/BIC and VaR.

        if dist in ["skewt", "skewstudent"]:
            eta = params.get("eta", params.get("nu", None))
            lam = params.get("lambda", None)
            if eta is not None and not (2.05 < eta < 200):
                return False, "bad_skewt_eta"
            # eta <= 4 indicates a fourth-moment warning for Hong-Lee inference,
            # but the volatility model itself can still be useful for AIC/BIC and VaR.
            if lam is not None and not (-0.995 < lam < 0.995):
                return False, "bad_skewt_lambda"

        # GED
        if dist == "ged":
            nu = params.get("nu", None)
            if nu is not None and not (1.01 < nu < 200):
                return False, "bad_ged_shape"

        # APARCH delta
        delta = params.get("delta", None)
        if delta is not None and not (0.05 < delta < 5.0):
            return False, "bad_aparch_delta"

        vol = spec.vol.lower()
        alpha_sum = _sum_params_by_prefix(params, ("alpha[",))
        beta_sum = _sum_params_by_prefix(params, ("beta[",))
        gamma_sum = _sum_params_by_prefix(params, ("gamma[",))

        # Hard persistence rejection is only applied to standard power=2 GARCH/GJR-style models.
        # For TARCH(power=1) and APARCH(delta estimated), the same proxy is not a valid stationarity condition;
        # those cases are handled as warnings downstream rather than hard fit failures.
        if vol == "garch" and abs(float(spec.power) - 2.0) < 1e-12:
            persistence_proxy = alpha_sum + beta_sum + 0.5 * max(gamma_sum, 0.0)
            if persistence_proxy >= 0.999:
                return False, f"near_integrated_volatility:persistence={persistence_proxy:.4f}"
            if persistence_proxy < -1e-8:
                return False, f"bad_negative_persistence_proxy:{persistence_proxy:.4f}"

        if vol == "egarch":
            max_abs_beta = max(
                [abs(float(v)) for k, v in params.items() if str(k).lower().startswith("beta[")] or [0.0]
            )
            if max_abs_beta >= 0.999:
                return False, f"near_integrated_egarch_beta:{max_abs_beta:.4f}"

        return True, "ok"

    except Exception as e:
        return False, f"param_check_error:{e}"


def check_hl_moment_condition(spec: ModelSpec, params: Dict[str, float]) -> Tuple[bool, str]:
    """
    Hong-Lee asymptotic inference uses high-order moment conditions.
    For t / skew-t innovations, df <= 4 is not a volatility-fit failure,
    but it should downgrade Hong-Lee health interpretation.
    """
    try:
        dist = spec.dist.lower()
        if dist in ["t", "studentst"]:
            nu = params.get("nu", None)
            if nu is not None and nu <= 4.05:
                return False, f"fourth_moment_warning:t_nu={nu:.4f}"
        if dist in ["skewt", "skewstudent"]:
            eta = params.get("eta", params.get("nu", None))
            if eta is not None and eta <= 4.05:
                return False, f"fourth_moment_warning:skewt_eta={eta:.4f}"
        return True, "ok"
    except Exception as e:
        return False, f"moment_check_error:{e}"


def check_persistence_warning(spec: ModelSpec, params: Dict[str, float]) -> Tuple[bool, str]:
    """
    Non-hard persistence warning for TARCH/APARCH-style models where the simple
    alpha + beta + 0.5 gamma proxy is not a valid exclusion rule.
    """
    try:
        vol = spec.vol.lower()
        alpha_sum = _sum_params_by_prefix(params, ("alpha[",))
        beta_sum = _sum_params_by_prefix(params, ("beta[",))
        gamma_sum = _sum_params_by_prefix(params, ("gamma[",))
        proxy = alpha_sum + beta_sum + 0.5 * max(gamma_sum, 0.0)

        if (vol == "garch" and abs(float(spec.power) - 1.0) < 1e-12) or vol == "aparch":
            if proxy >= 0.999:
                return True, f"review_persistence_proxy_not_hard_filter:{proxy:.4f}"
        return False, "ok"
    except Exception as e:
        return True, f"persistence_warning_error:{e}"


def fit_constvol_t(y: pd.Series) -> Dict[str, Any]:
    """
    可选的无条件 Student-t 厚尾基准。
    不是 GARCH 模型，只用于判断条件波动率模型是否有增量价值。
    """
    yy = pd.Series(y).dropna().astype(float)
    arr = yy.values
    n = len(arr)

    if n < 50:
        raise ValueError("constvol_t 样本不足。")

    mu0 = float(np.mean(arr))
    sigma0 = float(np.std(arr, ddof=1))
    nu0 = 8.0

    def unpack(theta):
        mu = theta[0]
        sigma = np.exp(theta[1])
        nu = 2.01 + np.exp(theta[2])
        return mu, sigma, nu

    def neg_ll(theta):
        mu, sigma, nu = unpack(theta)
        z = (arr - mu) / sigma
        ll = stats.t.logpdf(z, df=nu) - np.log(sigma)
        return -np.sum(ll)

    theta0 = np.array([mu0, np.log(max(sigma0, 1e-6)), np.log(nu0 - 2.01)])

    opt = minimize(
        neg_ll,
        theta0,
        method="Nelder-Mead",
        options={"maxiter": 5000, "xatol": 1e-8, "fatol": 1e-8},
    )

    mu, sigma, nu = unpack(opt.x)
    ll = -float(opt.fun)
    k = 3

    resid = yy - mu
    cond_vol = pd.Series(sigma, index=yy.index)
    std_resid = resid / sigma

    return {
        "success": bool(opt.success),
        "convergence_flag": 0 if opt.success else 1,
        "loglikelihood": ll,
        "aic": -2 * ll + 2 * k,
        "bic": -2 * ll + k * np.log(n),
        "params": {"mu": mu, "sigma": sigma, "nu": nu},
        "resid": resid,
        "std_resid": std_resid,
        "conditional_volatility": cond_vol,
        "conditional_mean": pd.Series(mu, index=yy.index),
        "nobs": n,
        "fit_object": None,
        "fit_warnings": "" if opt.success else str(opt.message)[:180],
    }


def fit_arch_spec(y: pd.Series, spec: ModelSpec) -> Dict[str, Any]:
    yy = pd.Series(y).dropna().astype(float)

    if len(yy) < MIN_OBS_MODEL:
        raise ValueError(f"样本不足：n={len(yy)}")

    if spec.model_type == "constvol_t":
        return fit_constvol_t(yy)

    if spec.mean == "AR":
        model = arch_model(
            yy,
            mean="AR",
            lags=spec.lags,
            vol=spec.vol,
            p=spec.p,
            o=spec.o,
            q=spec.q,
            power=spec.power,
            dist=spec.dist,
            rescale=False,
        )
    else:
        model = arch_model(
            yy,
            mean=spec.mean,
            vol=spec.vol,
            p=spec.p,
            o=spec.o,
            q=spec.q,
            power=spec.power,
            dist=spec.dist,
            rescale=False,
        )

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        res = model.fit(**ARCH_FIT_OPTIONS)

    fit_warnings = " | ".join(
        sorted({str(w.message)[:180] for w in caught if str(w.message)})
    )

    params = {str(k): float(v) for k, v in res.params.items()}
    resid = pd.Series(res.resid, index=yy.index)
    cond_vol = pd.Series(res.conditional_volatility, index=yy.index)
    std_resid = pd.Series(res.std_resid, index=yy.index)

    # 条件均值 = y - residual
    conditional_mean = yy - resid

    return {
        "success": True,
        "convergence_flag": int(getattr(res, "convergence_flag", 0)),
        "loglikelihood": float(res.loglikelihood),
        "aic": float(res.aic),
        "bic": float(res.bic),
        "params": params,
        "resid": resid,
        "std_resid": std_resid,
        "conditional_volatility": cond_vol,
        "conditional_mean": conditional_mean,
        "nobs": int(res.nobs),
        "fit_object": res,
        "fit_warnings": fit_warnings,
    }


In [6]:
def residual_diagnostics(std_resid: pd.Series) -> Dict[str, Any]:
    """
    传统模型诊断：
      1. 标准化残差 Ljung-Box
      2. 标准化残差平方 Ljung-Box
      3. ARCH-LM
    """
    z = pd.Series(std_resid).replace([np.inf, -np.inf], np.nan).dropna()

    if len(z) < 100:
        return {
            "diag_n": len(z),
            "lb_z_p_lag10": np.nan,
            "lb_z2_p_lag10": np.nan,
            "arch_lm_p_lag10": np.nan,
            "traditional_diag_pass": False,
        }

    lb_z = safe_ljungbox(z, [10])
    lb_z2 = safe_ljungbox(z ** 2, [10])
    arch_lm = safe_arch_lm(z, nlags=10)

    lb_z_p = lb_z.get("lb_p_lag10", np.nan)
    lb_z2_p = lb_z2.get("lb_p_lag10", np.nan)
    arch_p = arch_lm.get("arch_lm_p_lag10", np.nan)

    passed = (
        pd.notna(lb_z_p) and lb_z_p >= ALPHA and
        pd.notna(lb_z2_p) and lb_z2_p >= ALPHA and
        pd.notna(arch_p) and arch_p >= ALPHA
    )

    return {
        "diag_n": len(z),
        "lb_z_p_lag10": lb_z_p,
        "lb_z2_p_lag10": lb_z2_p,
        "arch_lm_p_lag10": arch_p,
        "traditional_diag_pass": bool(passed),
    }


def get_distribution_quantile(fit: Dict[str, Any], spec: ModelSpec, q: float, z_fallback: Optional[pd.Series] = None) -> float:
    """
    返回标准化残差分布的 q 分位数。
    arch 模型优先调用 arch distribution ppf。
    constvol_t 使用 scipy t。
    fallback 使用经验分位数。
    """
    try:
        if spec.model_type == "constvol_t":
            nu = fit["params"]["nu"]
            return float(stats.t.ppf(q, df=nu))

        res = fit["fit_object"]
        dist_obj = res.model.distribution
        param_names = list(dist_obj.parameter_names())
        params = []
        for name in param_names:
            if name in res.params.index:
                params.append(float(res.params[name]))

        params = np.asarray(params, dtype=float)
        return float(dist_obj.ppf(q, params))

    except Exception:
        if z_fallback is not None:
            z = pd.Series(z_fallback).replace([np.inf, -np.inf], np.nan).dropna()
            if len(z) > 30:
                return float(np.quantile(z, q))
        return float(stats.norm.ppf(q))


def safe_forecast_vol(fit: Dict[str, Any], spec: ModelSpec, horizon: int) -> float:
    """
    预测 horizon 日条件波动率。
    注意：部分模型多步解析预测可能不支持，自动 fallback 到 simulation。
    """
    try:
        if spec.model_type == "constvol_t":
            return float(fit["params"]["sigma"])

        res = fit["fit_object"]

        try:
            f = res.forecast(horizon=horizon, reindex=False)
        except Exception:
            f = res.forecast(horizon=horizon, reindex=False, method="simulation", simulations=1000)

        var = f.variance.iloc[-1, horizon - 1]
        return float(np.sqrt(var))

    except Exception:
        return np.nan


In [7]:
def var_backtest(y: pd.Series, fit: Dict[str, Any], spec: ModelSpec) -> List[Dict[str, Any]]:
    """
    多头 / 空头 95% 和 99% VaR 覆盖率检验。
    y 单位：百分比收益率。
    """
    yy = pd.Series(y).astype(float)

    mean = pd.Series(fit["conditional_mean"])
    sigma = pd.Series(fit["conditional_volatility"])
    z = pd.Series(fit["std_resid"])

    aligned = pd.concat(
        [
            yy.rename("ret"),
            mean.rename("mean"),
            sigma.rename("sigma"),
            z.rename("z"),
        ],
        axis=1,
    ).replace([np.inf, -np.inf], np.nan).dropna()

    out = []

    for level in VAR_LEVELS:
        tail_prob = 1.0 - level

        q_left = get_distribution_quantile(fit, spec, tail_prob, aligned["z"])
        q_right = get_distribution_quantile(fit, spec, level, aligned["z"])

        left_boundary = aligned["mean"] + aligned["sigma"] * q_left
        right_boundary = aligned["mean"] + aligned["sigma"] * q_right

        long_exceed = aligned["ret"] < left_boundary
        short_exceed = aligned["ret"] > right_boundary

        long_res = kupiec_pof_test(long_exceed.values, tail_prob)
        short_res = kupiec_pof_test(short_exceed.values, tail_prob)

        out.append({
            "side": "long",
            "backtest_method": "full_sample_in_sample",
            "var_level": level,
            "tail_prob": tail_prob,
            "var_boundary_latest_pct": float(left_boundary.iloc[-1]),
            **long_res,
        })

        out.append({
            "side": "short",
            "backtest_method": "full_sample_in_sample",
            "var_level": level,
            "tail_prob": tail_prob,
            "var_boundary_latest_pct": float(right_boundary.iloc[-1]),
            **short_res,
        })

    return out


def var_rolling_backtest(y: pd.Series, spec: ModelSpec) -> List[Dict[str, Any]]:
    """
    Expanding-window out-of-sample VaR backtest.

    为控制 20 个合约的总耗时，每隔 VAR_REFIT_EVERY 个观测重新估计一次，
    并对下一小段做 block forecast；这比全样本内覆盖率更接近真实回测。
    """
    yy = pd.Series(y).replace([np.inf, -np.inf], np.nan).dropna().astype(float)
    n = len(yy)
    train_min = max(MIN_OBS_MODEL, VAR_MIN_TRAIN_OBS, int(np.floor(n * VAR_ROLLING_INITIAL_FRACTION)))

    if n <= train_min + 30:
        fallback_fit = fit_arch_spec(yy, spec)
        rows = var_backtest(yy, fallback_fit, spec)
        for row in rows:
            row["backtest_method"] = "fallback_full_sample_sample_too_short"
        return rows

    records = []
    pos = train_min

    while pos < n:
        block = min(VAR_REFIT_EVERY, n - pos)
        train = yy.iloc[:pos]

        try:
            fit = fit_arch_spec(train, spec)
        except Exception:
            pos += block
            continue

        q_by_level = {}
        for level in VAR_LEVELS:
            tail_prob = 1.0 - level
            q_by_level[level] = (
                get_distribution_quantile(fit, spec, tail_prob, fit["std_resid"]),
                get_distribution_quantile(fit, spec, level, fit["std_resid"]),
            )

        if spec.model_type == "constvol_t":
            mean_fc = np.repeat(float(fit["params"]["mu"]), block)
            sigma_fc = np.repeat(float(fit["params"]["sigma"]), block)
        else:
            res = fit["fit_object"]
            try:
                fc = res.forecast(horizon=block, reindex=False)
            except Exception:
                fc = res.forecast(horizon=block, reindex=False, method="simulation", simulations=1000)
            mean_fc = np.asarray(fc.mean.iloc[-1, :block], dtype=float)
            sigma_fc = np.sqrt(np.asarray(fc.variance.iloc[-1, :block], dtype=float))

        for h in range(block):
            target_pos = pos + h
            ret = float(yy.iloc[target_pos])
            for level in VAR_LEVELS:
                q_left, q_right = q_by_level[level]
                left_boundary = float(mean_fc[h] + sigma_fc[h] * q_left)
                right_boundary = float(mean_fc[h] + sigma_fc[h] * q_right)
                records.append({
                    "target_date": yy.index[target_pos],
                    "var_level": level,
                    "tail_prob": 1.0 - level,
                    "ret": ret,
                    "left_boundary": left_boundary,
                    "right_boundary": right_boundary,
                    "long_exceed": ret < left_boundary,
                    "short_exceed": ret > right_boundary,
                })

        pos += block

    if not records:
        fallback_fit = fit_arch_spec(yy, spec)
        rows = var_backtest(yy, fallback_fit, spec)
        for row in rows:
            row["backtest_method"] = "fallback_full_sample_no_rolling_records"
        return rows

    bt = pd.DataFrame(records)
    out = []
    for level in VAR_LEVELS:
        level_rows = bt[bt["var_level"] == level]
        tail_prob = 1.0 - level
        long_res = kupiec_pof_test(level_rows["long_exceed"].values, tail_prob)
        short_res = kupiec_pof_test(level_rows["short_exceed"].values, tail_prob)

        out.append({
            "side": "long",
            "backtest_method": "expanding_refit_block_forecast",
            "var_level": level,
            "tail_prob": tail_prob,
            "var_boundary_latest_pct": float(level_rows["left_boundary"].iloc[-1]),
            "rolling_start": level_rows["target_date"].iloc[0],
            "rolling_end": level_rows["target_date"].iloc[-1],
            "refit_every": VAR_REFIT_EVERY,
            **long_res,
        })

        out.append({
            "side": "short",
            "backtest_method": "expanding_refit_block_forecast",
            "var_level": level,
            "tail_prob": tail_prob,
            "var_boundary_latest_pct": float(level_rows["right_boundary"].iloc[-1]),
            "rolling_start": level_rows["target_date"].iloc[0],
            "rolling_end": level_rows["target_date"].iloc[-1],
            "refit_every": VAR_REFIT_EVERY,
            **short_res,
        })

    return out


def var_recursive_one_step_backtest(y: pd.Series, spec: ModelSpec) -> List[Dict[str, Any]]:
    """
    Expanding-window fixed-parameter one-step VaR backtest.

    为控制计算量，默认每 VAR_REFIT_EVERY 个观测重新估计一次参数。
    在两次 refit 之间，不重新优化参数；但会用最新历史样本和固定参数重新过滤条件波动率，
    再做真实 1-step-ahead forecast。
    若 fixed filtering / one-step forecast 失败，则回退到从上次 refit 点出发的 h-step forecast，
    并在输出中记录 fallback 次数和占比。
    """
    yy = pd.Series(y).replace([np.inf, -np.inf], np.nan).dropna().astype(float)
    n = len(yy)
    train_min = max(MIN_OBS_MODEL, VAR_MIN_TRAIN_OBS, int(np.floor(n * VAR_ROLLING_INITIAL_FRACTION)))

    if n <= train_min + 30:
        fallback_fit = fit_arch_spec(yy, spec)
        rows = var_backtest(yy, fallback_fit, spec)
        for row in rows:
            row["backtest_method"] = "fallback_full_sample_sample_too_short_for_one_step"
            row["fixed_one_step_count"] = 0
            row["multistep_fallback_count"] = 0
            row["forecast_fallback_share"] = np.nan
        return rows

    records = []
    fit_cache = None
    last_refit_pos = train_min  # track last refit point for fallback h-step forecast
    fixed_one_step_count = 0
    multistep_fallback_count = 0

    for pos in range(train_min, n):
        if fit_cache is None or ((pos - train_min) % VAR_REFIT_EVERY == 0):
            try:
                fit_cache = fit_arch_spec(yy.iloc[:pos], spec)
                last_refit_pos = pos
            except Exception:
                fit_cache = None
                continue

        fit = fit_cache
        q_by_level = {}
        for level in VAR_LEVELS:
            tail_prob = 1.0 - level
            q_by_level[level] = (
                get_distribution_quantile(fit, spec, tail_prob, fit["std_resid"]),
                get_distribution_quantile(fit, spec, level, fit["std_resid"]),
            )

        forecast_mode = "fixed_parameter_one_step"
        if spec.model_type == "constvol_t":
            mean_fc = float(fit["params"]["mu"])
            sigma_fc = float(fit["params"]["sigma"])
            fixed_one_step_count += 1
        else:
            try:
                # True fixed-parameter 1-step-ahead forecast:
                # use observations only through pos-1, refilter with the last estimated parameters,
                # then forecast yy.iloc[pos] with horizon=1.
                if spec.mean == "AR":
                    am_temp = arch_model(
                        yy.iloc[:pos],
                        mean="AR",
                        lags=spec.lags,
                        vol=spec.vol,
                        p=spec.p,
                        o=spec.o,
                        q=spec.q,
                        power=spec.power,
                        dist=spec.dist,
                        rescale=False,
                    )
                else:
                    am_temp = arch_model(
                        yy.iloc[:pos],
                        mean=spec.mean,
                        vol=spec.vol,
                        p=spec.p,
                        o=spec.o,
                        q=spec.q,
                        power=spec.power,
                        dist=spec.dist,
                        rescale=False,
                    )

                res_fixed = am_temp.fix(fit["fit_object"].params)
                try:
                    fc = res_fixed.forecast(horizon=1, reindex=False)
                except Exception:
                    fc = res_fixed.forecast(horizon=1, reindex=False, method="simulation", simulations=1000)
                mean_fc = float(fc.mean.iloc[-1, 0])
                sigma_fc = float(np.sqrt(fc.variance.iloc[-1, 0]))
                fixed_one_step_count += 1
            except Exception:
                # Fallback only: use h-step forecast from the last refit point if fixed filtering fails.
                forecast_mode = "multistep_fallback"
                multistep_fallback_count += 1
                h = pos - last_refit_pos + 1
                res = fit["fit_object"]
                try:
                    fc = res.forecast(horizon=h, reindex=False)
                except Exception:
                    fc = res.forecast(horizon=h, reindex=False, method="simulation", simulations=1000)
                mean_fc = float(fc.mean.iloc[-1, h - 1])
                sigma_fc = float(np.sqrt(fc.variance.iloc[-1, h - 1]))

        ret = float(yy.iloc[pos])
        for level in VAR_LEVELS:
            q_left, q_right = q_by_level[level]
            left_boundary = mean_fc + sigma_fc * q_left
            right_boundary = mean_fc + sigma_fc * q_right
            records.append({
                "target_date": yy.index[pos],
                "var_level": level,
                "tail_prob": 1.0 - level,
                "ret": ret,
                "left_boundary": left_boundary,
                "right_boundary": right_boundary,
                "long_exceed": ret < left_boundary,
                "short_exceed": ret > right_boundary,
                "forecast_mode": forecast_mode,
            })

    if not records:
        fallback_fit = fit_arch_spec(yy, spec)
        rows = var_backtest(yy, fallback_fit, spec)
        for row in rows:
            row["backtest_method"] = "fallback_full_sample_no_one_step_records"
            row["fixed_one_step_count"] = fixed_one_step_count
            row["multistep_fallback_count"] = multistep_fallback_count
            total_count = fixed_one_step_count + multistep_fallback_count
            row["forecast_fallback_share"] = multistep_fallback_count / total_count if total_count > 0 else np.nan
        return rows

    bt = pd.DataFrame(records)
    out = []
    total_forecast_count = fixed_one_step_count + multistep_fallback_count
    forecast_fallback_share = (
        multistep_fallback_count / total_forecast_count
        if total_forecast_count > 0 else np.nan
    )
    for level in VAR_LEVELS:
        level_rows = bt[bt["var_level"] == level]
        tail_prob = 1.0 - level
        long_res = kupiec_pof_test(level_rows["long_exceed"].values, tail_prob)
        short_res = kupiec_pof_test(level_rows["short_exceed"].values, tail_prob)

        if VAR_REFIT_EVERY == 1:
            method_name = "expanding_daily_recursive_one_step"
        elif multistep_fallback_count > 0:
            method_name = f"expanding_refit_every_{VAR_REFIT_EVERY}_fixed_params_one_step_with_multistep_fallback"
        else:
            method_name = f"expanding_refit_every_{VAR_REFIT_EVERY}_fixed_params_one_step_forecast"

        out.append({
            "side": "long",
            "backtest_method": method_name,
            "var_level": level,
            "tail_prob": tail_prob,
            "var_boundary_latest_pct": float(level_rows["left_boundary"].iloc[-1]),
            "rolling_start": level_rows["target_date"].iloc[0],
            "rolling_end": level_rows["target_date"].iloc[-1],
            "refit_every": VAR_REFIT_EVERY,
            "fixed_one_step_count": fixed_one_step_count,
            "multistep_fallback_count": multistep_fallback_count,
            "forecast_fallback_share": forecast_fallback_share,
            **long_res,
        })
        out.append({
            "side": "short",
            "backtest_method": method_name,
            "var_level": level,
            "tail_prob": tail_prob,
            "var_boundary_latest_pct": float(level_rows["right_boundary"].iloc[-1]),
            "rolling_start": level_rows["target_date"].iloc[0],
            "rolling_end": level_rows["target_date"].iloc[-1],
            "refit_every": VAR_REFIT_EVERY,
            "fixed_one_step_count": fixed_one_step_count,
            "multistep_fallback_count": multistep_fallback_count,
            "forecast_fallback_share": forecast_fallback_share,
            **short_res,
        })

    return out


In [8]:
# Complete-var VaR version sanity check
# Run this cell after the VaR function cell. It should print True/True/True.
import inspect

_src = inspect.getsource(var_recursive_one_step_backtest)
_required_markers = [
    "fixed_one_step_count",
    "multistep_fallback_count",
    "forecast_fallback_share",
]
print("complete_var_markers:")
for _marker in _required_markers:
    print(f"{_marker}: {_marker in _src}")

_missing = [_marker for _marker in _required_markers if _marker not in _src]
assert not _missing, f"Current kernel is NOT using the complete_var VaR function. Missing: {_missing}"

print("OK: current kernel is using the complete_var VaR function.")


complete_var_markers:
fixed_one_step_count: True
multistep_fallback_count: True
forecast_fallback_share: True
OK: current kernel is using the complete_var VaR function.


In [9]:
def bartlett_weight(x: np.ndarray) -> np.ndarray:
    """
    Bartlett kernel: K(x) = 1 - |x| for |x| <= 1, else 0.
    """
    x = np.asarray(x)
    return np.where(np.abs(x) <= 1, 1 - np.abs(x), 0.0)


def default_hl_bandwidth(n: int) -> float:
    """
    Hong-Lee 带宽经验口径：p = 1.54 * n^(1/3)。
    """
    return 1.54 * (n ** (1.0 / 3.0))


def hl_weight_grid(grid_size: int = HL_WEIGHT_GRID_SIZE, scale: float = HL_WEIGHT_SCALE) -> Tuple[np.ndarray, np.ndarray]:
    """
    Approximate int g(v)dW(v), W=N(0, scale^2), by Gauss-Hermite quadrature.
    """
    if grid_size < 5:
        raise ValueError("HL_WEIGHT_GRID_SIZE must be at least 5.")
    x, w = np.polynomial.hermite.hermgauss(grid_size)
    v = np.sqrt(2.0) * scale * x
    weights = w / np.sqrt(np.pi)
    return v.astype(float), weights.astype(float)


def _prepare_hl_residuals(std_resid: pd.Series) -> Tuple[np.ndarray, Dict[str, float]]:
    z = pd.Series(std_resid).replace([np.inf, -np.inf], np.nan).dropna().astype(float).values
    meta = {
        "z_raw_mean": float(np.mean(z)) if len(z) else np.nan,
        "z_raw_second_moment": float(np.mean(z ** 2)) if len(z) else np.nan,
        "z_restandardized": bool(HL_RESTANDARDIZE_RESID),
    }
    if HL_RESTANDARDIZE_RESID and len(z) > 2:
        z = z - np.mean(z)
        s = np.sqrt(np.mean(z ** 2))
        if np.isfinite(s) and s > 0:
            z = z / s
    meta["z_used_mean"] = float(np.mean(z)) if len(z) else np.nan
    meta["z_used_second_moment"] = float(np.mean(z ** 2)) if len(z) else np.nan
    return z, meta


def hong_lee_kernel_test(
    std_resid: pd.Series,
    p: float,
    grid_size: int = HL_WEIGHT_GRID_SIZE,
    weight_scale: float = HL_WEIGHT_SCALE,
    compute_variance: bool = True,
) -> Dict[str, Any]:
    """
    Hong and Lee (2017) generalized spectral second-order derivative test.

    This implements the paper's core object:
      sigma_j^(2,0)(0,v) = cov(z_t^2, exp(i v z_{t-j}))
    estimated by centered characteristic-function transforms, then integrates
    |sigma_j^(2,0)(0,v)|^2 with respect to a symmetric W(v).
    """
    z, z_meta = _prepare_hl_residuals(std_resid)

    if len(z) < 200:
        return {
            "hl_n": len(z),
            "bandwidth_p": p,
            "max_lag": np.nan,
            "Q_gsd": np.nan,
            "TQ_gsd": np.nan,
            "M_robust": np.nan,
            "M_iid": np.nan,
            "pvalue": np.nan,
            "pvalue_iid": np.nan,
            "decision": "INSUFFICIENT_SAMPLE",
            **z_meta,
        }

    n = len(z)
    v_grid, w_grid = hl_weight_grid(grid_size=grid_size, scale=weight_scale)
    g = len(v_grid)

    all_lags = np.arange(1, n, dtype=int)
    lag_weights = bartlett_weight(all_lags.astype(float) / float(p))
    k2_values = lag_weights ** 2
    k4_values = lag_weights ** 4
    active_mask = k2_values > 1e-14
    active_lags = all_lags[active_mask]
    active_k2 = k2_values[active_mask]

    if len(active_lags) == 0:
        return {
            "hl_n": n,
            "bandwidth_p": float(p),
            "max_lag": 0,
            "Q_gsd": np.nan,
            "TQ_gsd": np.nan,
            "M_robust": np.nan,
            "M_iid": np.nan,
            "pvalue": np.nan,
            "pvalue_iid": np.nan,
            "decision": "BAD_BANDWIDTH",
            **z_meta,
        }

    # Precompute exp(i v z_t) once and reuse it for all lags/bandwidths.
    eiv_full = np.exp(1j * np.outer(z, v_grid))
    phi_full = eiv_full.mean(axis=0)
    psi_full = eiv_full - phi_full
    z4_minus_1_full = z ** 4 - 1.0
    sigma0_v_minus_v = 1.0 - np.abs(phi_full) ** 2
    int_sigma0_v_minus_v = float(np.sum(w_grid * np.real(sigma0_v_minus_v)))

    # Pre-compute full-sample fourth moment used in both c_robust and c_iid.
    z4_mean_minus_1 = float(np.mean(z ** 4) - 1.0)
    q_gsd = 0.0
    c_robust = 0.0

    for idx, j in enumerate(active_lags):
        k2 = float(active_k2[idx])
        z_current = z[j:]
        eiv_lagged = eiv_full[:-j, :]
        n_j = n - j

        sigma_j = np.mean(
            (z_current ** 2 - 1.0)[:, None] * (eiv_lagged - phi_full),
            axis=0,
        )
        int_sigma2 = float(np.sum(w_grid * (np.abs(sigma_j) ** 2)))
        q_gsd += k2 * n_j * int_sigma2

        # Use full-sample E[z^4-1] (not sub-sample z_current) per paper definition.
        c_robust += k2 * z4_mean_minus_1 * int_sigma0_v_minus_v

    # Paper-grade robust scaling: upper-triangle sum over lag pairs (j,l).
    # Diagonal pairs are counted once; off-diagonal pairs are counted twice.
    # Skipped when compute_variance=False (candidate screening), saving O(L^2*n*g^2) cost.
    if compute_variance:
        d2_robust = 0.0
        for a, j in enumerate(active_lags):
            k2_j = float(active_k2[a])
            for b in range(a, len(active_lags)):
                ell = active_lags[b]
                k2_l = float(active_k2[b])
                symmetry = 1.0 if a == b else 2.0
                m = int(max(j, ell))
                n_m = n - m
                if n_m <= 0:
                    continue

                weight_t = z4_minus_1_full[m:]
                psi_j_u = psi_full[m - j:n - j, :]
                psi_l_v = psi_full[m - ell:n - ell, :]

                a_uv = np.einsum(
                    "t,tu,tv->uv",
                    weight_t,
                    psi_j_u,
                    np.conjugate(psi_l_v),
                    optimize=True,
                ) / n_m

                int_abs_a_sq = np.sum(
                    w_grid[:, None] * w_grid[None, :] * np.abs(a_uv) ** 2
                )
                d2_robust += symmetry * k2_j * k2_l * float(int_abs_a_sq)

        d_robust = np.sqrt(2.0 * d2_robust) if d2_robust > 0 and np.isfinite(d2_robust) else np.nan
    else:
        d_robust = np.nan

    uv_sum_grid = v_grid[:, None] + v_grid[None, :]
    phi_uv = np.exp(1j * np.outer(z, uv_sum_grid.ravel())).mean(axis=0).reshape(g, g)
    sigma0_uv = phi_uv - phi_full[:, None] * phi_full[None, :]
    int_abs_sigma0_uv_sq = float(
        np.sum(w_grid[:, None] * w_grid[None, :] * np.abs(sigma0_uv) ** 2)
    )

    c_iid = z4_mean_minus_1 * int_sigma0_v_minus_v * float(np.sum(active_k2))
    d2_iid = (
        2.0
        * (z4_mean_minus_1 ** 2)
        * int_abs_sigma0_uv_sq
        * float(np.sum(active_k2 ** 2))
    )
    d_iid = np.sqrt(d2_iid) if d2_iid > 0 and np.isfinite(d2_iid) else np.nan

    if np.isfinite(d_robust) and d_robust > 0:
        m_robust = (q_gsd - c_robust) / d_robust
        pvalue = 1.0 - stats.norm.cdf(m_robust)
        decision = "REJECT" if pvalue < ALPHA else "FAIL_TO_REJECT"
    elif not compute_variance:
        m_robust = np.nan
        pvalue = np.nan
        decision = "VARIANCE_NOT_COMPUTED"
    else:
        m_robust = np.nan
        pvalue = np.nan
        decision = "BAD_VARIANCE"

    if np.isfinite(d_iid) and d_iid > 0:
        m_iid = (q_gsd - c_iid) / d_iid
        pvalue_iid = 1.0 - stats.norm.cdf(m_iid)
    else:
        m_iid = np.nan
        pvalue_iid = np.nan

    return {
        "hl_n": n,
        "bandwidth_p": float(p),
        "max_lag": int(active_lags.max()),
        "active_lag_count": int(len(active_lags)),
        "hl_weight_grid_size": int(grid_size),
        "hl_weight_scale": float(weight_scale),
        "Q_gsd": q_gsd,
        "TQ_gsd": q_gsd,
        "C_robust": c_robust,
        "D_robust": float(d_robust) if np.isfinite(d_robust) else np.nan,
        "M_robust": float(m_robust) if np.isfinite(m_robust) else np.nan,
        "pvalue": float(pvalue) if np.isfinite(pvalue) else np.nan,
        "C_iid": c_iid,
        "D_iid": float(d_iid) if np.isfinite(d_iid) else np.nan,
        "M_iid": float(m_iid) if np.isfinite(m_iid) else np.nan,
        "pvalue_iid": float(pvalue_iid) if np.isfinite(pvalue_iid) else np.nan,
        "decision": decision,
        **z_meta,
    }


In [10]:
def adjust_pvalues(pvalues: pd.Series, method: str = MULTIPLE_TEST_METHOD) -> pd.Series:
    p = pd.Series(pvalues, dtype=float)
    valid = p.notna()
    out = pd.Series(np.nan, index=p.index, dtype=float)
    vals = p[valid].values
    m = len(vals)
    if m == 0:
        return out

    order = np.argsort(vals)
    sorted_vals = vals[order]
    adjusted = np.empty(m, dtype=float)

    if method.lower() == "holm":
        running = 0.0
        for rank, pv in enumerate(sorted_vals, start=1):
            running = max(running, (m - rank + 1) * pv)
            adjusted[rank - 1] = min(running, 1.0)
    else:
        running = 1.0
        for idx in range(m - 1, -1, -1):
            rank = idx + 1
            running = min(running, sorted_vals[idx] * m / rank)
            adjusted[idx] = min(running, 1.0)

    restored = np.empty(m, dtype=float)
    restored[order] = adjusted
    out.loc[valid] = restored
    return out


def summarize_hl_decisions(rows: pd.DataFrame) -> Dict[str, Any]:
    valid = rows[rows["decision"].isin(["REJECT", "FAIL_TO_REJECT"])].copy()

    if valid.empty:
        return {
            "reject_count": np.nan,
            "valid_bandwidth_count": 0,
            "reject_share": np.nan,
            "bandwidth_stability": "NO_VALID_TEST",
        }

    reject_count = int((valid["decision"] == "REJECT").sum())
    total = len(valid)
    reject_share = reject_count / total

    if reject_share == 0:
        stability = "stable_fail_to_reject"
    elif reject_share == 1:
        stability = "stable_reject"
    else:
        stability = "bandwidth_sensitive"

    return {
        "reject_count": reject_count,
        "valid_bandwidth_count": total,
        "reject_share": reject_share,
        "bandwidth_stability": stability,
    }


def classify_model_health_from_hl(
    n: int,
    traditional_pass: bool,
    fit_warning_flag: bool,
    hl_moment_condition_ok: bool,
    persistence_warning_flag: bool,
    hl_decision: str,
    bandwidth_stability: str,
) -> str:
    if n < MIN_OBS_MODEL:
        return "low_sample"
    if fit_warning_flag:
        return "fit_warning_review_needed"
    if not traditional_pass:
        return "traditional_diag_failed"
    if not hl_moment_condition_ok:
        return "hl_inference_limited_fourth_moment"
    if persistence_warning_flag:
        return "persistence_warning_review_needed"
    if hl_decision == "REJECT":
        return "hong_lee_rejected"
    if bandwidth_stability == "bandwidth_sensitive":
        return "borderline_bandwidth_sensitive"
    if hl_decision == "FAIL_TO_REJECT" and bandwidth_stability == "stable_fail_to_reject":
        return "healthy"
    if hl_decision == "FAIL_TO_REJECT":
        return "borderline"
    return "invalid_or_borderline"


def run_hl_sensitivity_grid(std_resid: pd.Series, p0: float, label: str) -> pd.DataFrame:
    """
    One-at-a-time sensitivity report for bandwidth, W(v) grid size, and W(v) scale.
    """
    rows = []
    seen = set()

    candidates = []
    for mult in HL_BANDWIDTH_MULTIPLIERS:
        candidates.append({
            "sensitivity_axis": "bandwidth_multiplier",
            "bandwidth_multiplier": mult,
            "p": p0 * mult,
            "grid_size": HL_WEIGHT_GRID_SIZE,
            "weight_scale": HL_WEIGHT_SCALE,
        })
    for grid_size in HL_SENSITIVITY_GRID_SIZES:
        candidates.append({
            "sensitivity_axis": "weight_grid_size",
            "bandwidth_multiplier": 1.0,
            "p": p0,
            "grid_size": grid_size,
            "weight_scale": HL_WEIGHT_SCALE,
        })
    for scale in HL_SENSITIVITY_WEIGHT_SCALES:
        candidates.append({
            "sensitivity_axis": "weight_scale",
            "bandwidth_multiplier": 1.0,
            "p": p0,
            "grid_size": HL_WEIGHT_GRID_SIZE,
            "weight_scale": scale,
        })

    for cand in candidates:
        key = (
            cand["sensitivity_axis"],
            round(float(cand["p"]), 10),
            int(cand["grid_size"]),
            round(float(cand["weight_scale"]), 10),
        )
        if key in seen:
            continue
        seen.add(key)
        try:
            res = hong_lee_kernel_test(
                std_resid,
                cand["p"],
                grid_size=int(cand["grid_size"]),
                weight_scale=float(cand["weight_scale"]),
            )
            status = "ok"
        except Exception as e:
            res = {
                "pvalue": np.nan,
                "M_robust": np.nan,
                "decision": "ERROR",
                "error": str(e)[:300],
            }
            status = "error"
        res.update(cand)
        res["sensitivity_label"] = label
        res["status"] = status
        rows.append(res)

    out = pd.DataFrame(rows)
    valid = out[out["decision"].isin(["REJECT", "FAIL_TO_REJECT"])].copy()
    if not valid.empty:
        reject_share = float((valid["decision"] == "REJECT").mean())
        if reject_share == 0:
            conclusion = "stable_fail_to_reject"
        elif reject_share == 1:
            conclusion = "stable_reject"
        else:
            conclusion = "sensitive"
        out["sensitivity_reject_share"] = reject_share
        out["sensitivity_conclusion"] = conclusion
    else:
        out["sensitivity_reject_share"] = np.nan
        out["sensitivity_conclusion"] = "no_valid_test"
    return out


def build_hl_bandwidth_candidates(
    n: int,
    c_grid: List[float] = HL_DD_C_GRID,
    lambda_grid: List[float] = HL_DD_LAMBDA_GRID,
    min_p: float = 3.0,
    max_ratio: float = HL_DD_MAX_RATIO,
) -> pd.DataFrame:
    rows = []
    for c in c_grid:
        for lam in lambda_grid:
            p = c * (n ** lam)
            p = max(min_p, p)
            p = min(p, n * max_ratio)
            rows.append({"c": c, "lambda": lam, "p": float(p)})
    out = pd.DataFrame(rows)
    out["p_rounded"] = out["p"].round(6)
    return out.drop_duplicates("p_rounded").reset_index(drop=True)


def run_hl_data_driven_bandwidth(std_resid: pd.Series, label: str) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    z = pd.Series(std_resid).replace([np.inf, -np.inf], np.nan).dropna()
    n = len(z)
    candidates = build_hl_bandwidth_candidates(n)
    rows = []
    for _, cand in candidates.iterrows():
        try:
            res = hong_lee_kernel_test(
                z,
                float(cand["p"]),
                grid_size=HL_WEIGHT_GRID_SIZE,
                weight_scale=HL_WEIGHT_SCALE,
            )
            status = "ok"
        except Exception as e:
            res = {
                "M_robust": np.nan,
                "pvalue": np.nan,
                "decision": "ERROR",
                "error": str(e)[:300],
            }
            status = "error"
        res.update({
            "dd_label": label,
            "c": cand["c"],
            "lambda": cand["lambda"],
            "candidate_p": cand["p"],
            "status": status,
        })
        rows.append(res)

    results = pd.DataFrame(rows)
    ok = results[
        (results["status"] == "ok") &
        (results["M_robust"].notna()) &
        (results["pvalue"].notna())
    ].copy()
    if ok.empty:
        selected = {
            "dd_label": label,
            "dd_status": "no_valid_candidate",
            "dd_selected_p": np.nan,
            "dd_selected_M_robust": np.nan,
            "dd_selected_pvalue": np.nan,
            "dd_post_selection_boundary": "selected-p pvalue is descriptive",
        }
        return results, selected

    ok["dd_penalty"] = (
        HL_DD_PENALTY_STRENGTH
        * np.sqrt(ok["candidate_p"] / n)
        * np.sqrt(np.log(n))
    )
    ok["dd_objective"] = ok["M_robust"] - ok["dd_penalty"]
    ok = ok.sort_values("dd_objective", ascending=False).reset_index(drop=True)
    pick = ok.iloc[0]

    selected = {
        "dd_label": label,
        "dd_status": "ok",
        "dd_selected_p": float(pick["candidate_p"]),
        "dd_selected_c": float(pick["c"]),
        "dd_selected_lambda": float(pick["lambda"]),
        "dd_selected_M_robust": float(pick["M_robust"]),
        "dd_selected_pvalue": float(pick["pvalue"]),
        "dd_selected_decision": pick["decision"],
        "dd_selected_objective": float(pick["dd_objective"]),
        "dd_candidate_count": int(len(results)),
        "dd_valid_candidate_count": int(len(ok)),
        "dd_post_selection_boundary": "selected-p pvalue is descriptive; use MC for finite-sample reliability",
    }
    return results, selected


def simulate_from_arch_fit(fit: Dict[str, Any], nobs: int) -> Optional[pd.Series]:
    try:
        res = fit["fit_object"]
        sim = res.model.simulate(res.params, nobs=nobs, burn=HL_MC_BURN)
        if isinstance(sim, pd.DataFrame) and "data" in sim.columns:
            return pd.Series(sim["data"].values)
        if isinstance(sim, dict) and "data" in sim:
            return pd.Series(sim["data"])
        return pd.Series(np.asarray(sim).ravel())
    except Exception:
        return None


def run_hl_monte_carlo_size_for_family(
    symbol: str,
    family: str,
    spec: ModelSpec,
    fit: Dict[str, Any],
    reps: int = HL_MC_REPS,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    rng_state = np.random.get_state()
    np.random.seed(stable_int_seed(symbol, family))

    z0 = pd.Series(fit["std_resid"]).replace([np.inf, -np.inf], np.nan).dropna()
    nobs = len(z0)
    p0 = default_hl_bandwidth(nobs)
    rows = []

    for rep in range(1, reps + 1):
        row = {
            "symbol": symbol,
            "family": family,
            "spec_name": spec.name,
            "rep": rep,
            "nobs": nobs,
            "bandwidth_p": p0,
        }
        y_sim = simulate_from_arch_fit(fit, nobs=nobs)
        if y_sim is None:
            row.update({"status": "simulate_failed", "pvalue": np.nan, "reject_5pct": np.nan})
            rows.append(row)
            continue

        try:
            fit_sim = fit_arch_spec(y_sim, spec)
            hl = hong_lee_kernel_test(fit_sim["std_resid"], p0)
            row.update({
                "status": "ok",
                "convergence_flag": fit_sim.get("convergence_flag", np.nan),
                "M_robust": hl.get("M_robust"),
                "pvalue": hl.get("pvalue"),
                "decision": hl.get("decision"),
                "reject_5pct": bool(hl.get("pvalue", np.nan) < ALPHA) if pd.notna(hl.get("pvalue", np.nan)) else np.nan,
            })
        except Exception as e:
            row.update({
                "status": f"fit_or_test_failed:{str(e)[:180]}",
                "pvalue": np.nan,
                "reject_5pct": np.nan,
            })
        rows.append(row)

    np.random.set_state(rng_state)

    detail = pd.DataFrame(rows)
    ok = detail[detail["status"] == "ok"].copy()
    summary = {
        "symbol": symbol,
        "family": family,
        "spec_name": spec.name,
        "requested_reps": reps,
        "ok_reps": int(len(ok)),
        "rejection_rate_5pct": float(ok["reject_5pct"].mean()) if len(ok) else np.nan,
        "mean_pvalue": float(ok["pvalue"].mean()) if len(ok) else np.nan,
        "mc_size_target": ALPHA,
        "mc_size_abs_error": float(abs(ok["reject_5pct"].mean() - ALPHA)) if len(ok) else np.nan,
    }
    return detail, summary



In [11]:
# 一、模型筛选
# ============================================================

print("\n" + "=" * 80)
print("一、模型筛选")
print("=" * 80)

df_raw = pd.read_csv(INPUT_PATH)

required_cols = [
    "date", "symbol", "close", "log_ret_1d",
    "valid_return_flag", "tradable_return_flag",
]
missing = [c for c in required_cols if c not in df_raw.columns]
if missing:
    raise KeyError(f"输入数据缺少必要字段：{missing}")

all_candidate_rows = []
best_model_rows = []
fit_store: Dict[Tuple[str, str], Dict[str, Any]] = {}
data_store: Dict[str, pd.DataFrame] = {}

start_time = time.time()

for symbol in HIGH_RELIABILITY_SYMBOLS:
    print(f"\n--- Fitting symbol: {symbol} ---")

    sub = clean_symbol_data(df_raw, symbol)
    data_store[symbol] = sub

    y = sub.set_index("date")["ret_pct"]

    symbol_n = len(y)
    symbol_start = sub["date"].min()
    symbol_end = sub["date"].max()

    if symbol_n < MIN_OBS_MODEL:
        print(f"Skip {symbol}: sample too short, n={symbol_n}")
        continue

    symbol_results = []

    for spec in MODEL_SPECS:
        row = {
            "symbol": symbol,
            "model_name": spec.name,
            "model_type": spec.model_type,
            "mean": spec.mean,
            "lags": spec.lags,
            "vol": spec.vol,
            "p": spec.p,
            "o": spec.o,
            "q": spec.q,
            "power": spec.power,
            "dist": spec.dist,
            "sample_start": symbol_start,
            "sample_end": symbol_end,
            "sample_size": symbol_n,
            "reliability": reliability_by_n(symbol_n),
        }

        try:
            fit = fit_arch_spec(y, spec)
            params_ok, param_reason = check_params_ok(spec, fit["params"])
            hl_moment_ok, hl_moment_reason = check_hl_moment_condition(spec, fit["params"])
            persistence_warning_flag, persistence_warning_reason = check_persistence_warning(spec, fit["params"])
            diag = residual_diagnostics(fit["std_resid"])
            p0_model = default_hl_bandwidth(len(pd.Series(fit["std_resid"]).dropna()))
            hl_model = hong_lee_kernel_test(fit["std_resid"], p0_model, compute_variance=False)

            convergence_ok = fit["convergence_flag"] == 0
            success = bool(fit["success"] and convergence_ok and params_ok)

            row.update({
                "fit_success": bool(fit["success"]),
                "convergence_flag": fit["convergence_flag"],
                "convergence_ok": bool(convergence_ok),
                "params_ok": bool(params_ok),
                "param_check_reason": param_reason,
                "hl_moment_condition_ok": bool(hl_moment_ok),
                "hl_moment_condition_reason": hl_moment_reason,
                "persistence_warning_flag": bool(persistence_warning_flag),
                "persistence_warning_reason": persistence_warning_reason,
                "loglikelihood": fit["loglikelihood"],
                "aic": fit["aic"],
                "bic": fit["bic"],
                "nobs_model": fit["nobs"],
                "params_json": params_to_json(fit["params"]),
                "fit_warnings": fit.get("fit_warnings", ""),
                "fit_warning_flag": is_material_fit_warning(fit.get("fit_warnings", "")),
                "model_usable": bool(success),
            })
            row.update(diag)
            for hk, hv in hl_model.items():
                row[f"hl_{hk}"] = hv

            fit_store[(symbol, spec.name)] = fit

        except Exception as e:
            row.update({
                "fit_success": False,
                "convergence_flag": np.nan,
                "convergence_ok": False,
                "params_ok": False,
                "param_check_reason": "fit_error",
                "hl_moment_condition_ok": False,
                "hl_moment_condition_reason": "fit_error",
                "persistence_warning_flag": False,
                "persistence_warning_reason": "fit_error",
                "loglikelihood": np.nan,
                "aic": np.nan,
                "bic": np.nan,
                "nobs_model": np.nan,
                "params_json": "{}",
                "fit_warnings": "",
                "fit_warning_flag": False,
                "model_usable": False,
                "diag_n": np.nan,
                "lb_z_p_lag10": np.nan,
                "lb_z2_p_lag10": np.nan,
                "arch_lm_p_lag10": np.nan,
                "traditional_diag_pass": False,
                "hl_decision": "FIT_ERROR",
                "hl_pvalue": np.nan,
                "hl_M_robust": np.nan,
                "error": str(e)[:300],
            })

        symbol_results.append(row)

    res_df = pd.DataFrame(symbol_results)

    if "hl_pvalue" in res_df.columns:
        res_df["hl_pvalue_adj_symbol"] = adjust_pvalues(res_df["hl_pvalue"])
        res_df["hl_decision_adj_symbol"] = np.where(
            res_df["hl_pvalue_adj_symbol"].notna() & (res_df["hl_pvalue_adj_symbol"] < ALPHA),
            "REJECT",
            np.where(res_df["hl_pvalue_adj_symbol"].notna(), "FAIL_TO_REJECT", res_df.get("hl_decision", np.nan)),
        )

    all_candidate_rows.extend(res_df.to_dict("records"))

    # best_accepted_model:
    # 主模型选择只使用 fit usable + 参数合理 + 传统诊断通过 + BIC/AIC。
    # Hong-Lee 保留为 post-selection adequacy / health classification，
    # 避免把同一检验同时用于筛选和确认而造成 post-selection p-value 解释污染。
    accepted = res_df[
        (res_df["model_usable"]) &
        (res_df["traditional_diag_pass"]) &
        (res_df["bic"].notna()) &
        (res_df["aic"].notna())
    ].copy()

    if len(accepted) > 0:
        sort_col = MODEL_SELECTION_CRITERION
        best = accepted.sort_values(sort_col, ascending=True).iloc[0].to_dict()
        best_status = "accepted_by_traditional_diagnostics_hl_post_check"
    else:
        # 若没有模型通过传统诊断，则在可用模型里按 BIC 选一个 fallback
        usable = res_df[
            (res_df["model_usable"]) &
            (res_df["bic"].notna()) &
            (res_df["aic"].notna())
        ].copy()

        if len(usable) > 0:
            best = usable.sort_values(MODEL_SELECTION_CRITERION, ascending=True).iloc[0].to_dict()
            best_status = "fallback_usable_but_diagnostics_failed"
        else:
            best = res_df.sort_values("bic", ascending=True, na_position="last").iloc[0].to_dict()
            best_status = "fallback_no_usable_model"

    best_row = {
        "symbol": symbol,
        "best_model_name": best.get("model_name"),
        "best_status": best_status,
        "selection_criterion": MODEL_SELECTION_CRITERION,
        "sample_start": symbol_start,
        "sample_end": symbol_end,
        "sample_size": symbol_n,
        "reliability": reliability_by_n(symbol_n),
        "best_aic": best.get("aic"),
        "best_bic": best.get("bic"),
        "best_loglikelihood": best.get("loglikelihood"),
        "traditional_diag_pass": best.get("traditional_diag_pass"),
        "lb_z_p_lag10": best.get("lb_z_p_lag10"),
        "lb_z2_p_lag10": best.get("lb_z2_p_lag10"),
        "arch_lm_p_lag10": best.get("arch_lm_p_lag10"),
        "hl_screen_decision": best.get("hl_decision"),
        "hl_screen_pvalue": best.get("hl_pvalue"),
        "hl_screen_pvalue_adj_symbol": best.get("hl_pvalue_adj_symbol"),
        "hl_screen_M_robust": best.get("hl_M_robust"),
        "hl_screen_M_iid": best.get("hl_M_iid"),
        "hl_moment_condition_ok": best.get("hl_moment_condition_ok"),
        "hl_moment_condition_reason": best.get("hl_moment_condition_reason"),
        "persistence_warning_flag": best.get("persistence_warning_flag"),
        "persistence_warning_reason": best.get("persistence_warning_reason"),
        "fit_warnings": best.get("fit_warnings"),
        "fit_warning_flag": best.get("fit_warning_flag"),
        "params_json": best.get("params_json"),
    }

    # 同时记录 purely best AIC / BIC
    success_models = res_df[
        (res_df["model_usable"]) &
        (res_df["aic"].notna()) &
        (res_df["bic"].notna())
    ].copy()

    if len(success_models) > 0:
        best_aic_row = success_models.sort_values("aic").iloc[0]
        best_bic_row = success_models.sort_values("bic").iloc[0]
        best_row["best_by_aic"] = best_aic_row["model_name"]
        best_row["best_by_bic"] = best_bic_row["model_name"]
    else:
        best_row["best_by_aic"] = None
        best_row["best_by_bic"] = None

    best_model_rows.append(best_row)

    print(
        f"{symbol}: n={symbol_n}, best={best_row['best_model_name']}, "
        f"status={best_status}, BIC={best_row['best_bic']}"
    )

candidate_results = pd.DataFrame(all_candidate_rows)
best_models = pd.DataFrame(best_model_rows)

candidate_results.to_csv(OUTPUT_DIR / "01_model_selection_all_candidates.csv", index=False, encoding="utf-8-sig")
best_models.to_csv(OUTPUT_DIR / "02_best_model_by_symbol.csv", index=False, encoding="utf-8-sig")

print("\n模型筛选完成。")
print("候选模型结果:", OUTPUT_DIR / "01_model_selection_all_candidates.csv")
print("最佳模型结果:", OUTPUT_DIR / "02_best_model_by_symbol.csv")


# ============================================================


一、模型筛选

--- Fitting symbol: AG ---
AG: n=2537, best=08_EGARCH11_t, status=accepted_by_traditional_diagnostics_hl_post_check, BIC=8806.613154401517

--- Fitting symbol: AL ---
AL: n=2537, best=05_GARCH11_ged, status=accepted_by_traditional_diagnostics_hl_post_check, BIC=7076.2741464121655

--- Fitting symbol: AU ---
AU: n=2537, best=08_EGARCH11_t, status=accepted_by_traditional_diagnostics_hl_post_check, BIC=5876.515529119723

--- Fitting symbol: CF ---
CF: n=2537, best=10_EGARCH21_t, status=accepted_by_traditional_diagnostics_hl_post_check, BIC=7478.156671705052

--- Fitting symbol: CU ---
CU: n=2537, best=19_AR1_EGARCH11_t, status=accepted_by_traditional_diagnostics_hl_post_check, BIC=7063.268133957694

--- Fitting symbol: FG ---
FG: n=2537, best=19_AR1_EGARCH11_t, status=accepted_by_traditional_diagnostics_hl_post_check, BIC=9734.281872785226

--- Fitting symbol: HC ---
HC: n=2537, best=19_AR1_EGARCH11_t, status=accepted_by_traditional_diagnostics_hl_post_check, BIC=8744.23174695485

In [12]:
# 二、传统模型检验
# ============================================================

print("\n" + "=" * 80)
print("二、传统模型检验")
print("=" * 80)

return_diag_rows = []
best_diag_rows = []
var_rows = []
vol_state_rows = []

for symbol in HIGH_RELIABILITY_SYMBOLS:
    if symbol not in data_store:
        continue

    sub = data_store[symbol].copy()
    y = sub.set_index("date")["ret_pct"]

    # ----------------------------
    # 2.1 原始收益率诊断
    # ----------------------------
    adf_res = safe_adf(y)
    lb_ret = safe_ljungbox(y, [5, 10, 20, 30])
    lb_ret2 = safe_ljungbox(y ** 2, [5, 10, 20, 30])
    arch5 = safe_arch_lm(y, nlags=5)
    arch10 = safe_arch_lm(y, nlags=10)
    arch20 = safe_arch_lm(y, nlags=20)

    desc = {
        "symbol": symbol,
        "sample_start": sub["date"].min(),
        "sample_end": sub["date"].max(),
        "sample_size": len(y),
        "reliability": reliability_by_n(len(y)),
        "mean_pct": float(y.mean()),
        "std_pct": float(y.std(ddof=1)),
        "min_pct": float(y.min()),
        "max_pct": float(y.max()),
        "q01_pct": float(y.quantile(0.01)),
        "q05_pct": float(y.quantile(0.05)),
        "q95_pct": float(y.quantile(0.95)),
        "q99_pct": float(y.quantile(0.99)),
        "skew": float(y.skew()),
        "excess_kurtosis": float(y.kurt()),
        "raw_kurtosis": float(y.kurt() + 3),
    }

    desc.update(adf_res)

    for k, v in lb_ret.items():
        desc[f"ret_{k}"] = v
    for k, v in lb_ret2.items():
        desc[f"ret2_{k}"] = v

    desc.update(arch5)
    desc.update(arch10)
    desc.update(arch20)

    return_diag_rows.append(desc)

    # ----------------------------
    # 2.2 最优模型残差诊断、VaR、波动率状态
    # ----------------------------
    best_row = best_models[best_models["symbol"] == symbol]
    if best_row.empty:
        continue

    best_model_name = best_row["best_model_name"].iloc[0]
    spec = next((s for s in MODEL_SPECS if s.name == best_model_name), None)

    if spec is None:
        continue

    fit = fit_store.get((symbol, best_model_name), None)
    if fit is None:
        continue

    diag = residual_diagnostics(fit["std_resid"])
    diag_row = {
        "symbol": symbol,
        "best_model_name": best_model_name,
        "best_status": best_row["best_status"].iloc[0],
        "sample_size": len(y),
        "reliability": reliability_by_n(len(y)),
    }
    diag_row.update(diag)
    best_diag_rows.append(diag_row)

    # VaR 回测
    try:
        if ROLLING_VAR_BACKTEST and VAR_ONE_STEP_RECURSIVE:
            var_out = var_recursive_one_step_backtest(y, spec)
        elif ROLLING_VAR_BACKTEST:
            var_out = var_rolling_backtest(y, spec)
        else:
            var_out = var_backtest(y, fit, spec)
        for r in var_out:
            r.update({
                "symbol": symbol,
                "best_model_name": best_model_name,
                "sample_size": len(y),
                "reliability": reliability_by_n(len(y)),
            })
            var_rows.append(r)
    except Exception as e:
        var_rows.append({
            "symbol": symbol,
            "best_model_name": best_model_name,
            "error": str(e)[:300],
        })

    # 当前波动率状态
    cond_vol = pd.Series(fit["conditional_volatility"]).replace([np.inf, -np.inf], np.nan).dropna()
    cond_mean = pd.Series(fit["conditional_mean"]).replace([np.inf, -np.inf], np.nan).dropna()

    if len(cond_vol) > 30:
        latest_date = sub["date"].max()
        latest_close = float(sub.sort_values("date")["close"].iloc[-1])
        latest_ret = float(y.iloc[-1])

        latest_vol = float(cond_vol.iloc[-1])
        vol_pctile = float((cond_vol <= latest_vol).mean())

        median_vol = float(cond_vol.median())
        raw_scale = median_vol / latest_vol if latest_vol > 0 else np.nan
        capped_scale = float(np.clip(raw_scale, 0.30, 1.50)) if pd.notna(raw_scale) else np.nan

        vol_state_rows.append({
            "symbol": symbol,
            "best_model_name": best_model_name,
            "latest_date": latest_date,
            "latest_close": latest_close,
            "latest_return_pct": latest_ret,
            "latest_cond_vol_pct": latest_vol,
            "median_cond_vol_pct": median_vol,
            "vol_percentile": vol_pctile,
            "vol_state": classify_vol_state(vol_pctile),
            "position_scale_raw": raw_scale,
            "position_scale_capped": capped_scale,
            "forecast_vol_1d_pct": safe_forecast_vol(fit, spec, 1),
            "forecast_vol_5d_pct": safe_forecast_vol(fit, spec, 5),
            "forecast_vol_10d_pct": safe_forecast_vol(fit, spec, 10),
            "sample_size": len(y),
            "reliability": reliability_by_n(len(y)),
        })

return_diagnostics = pd.DataFrame(return_diag_rows)
best_residual_diagnostics = pd.DataFrame(best_diag_rows)
var_backtests = pd.DataFrame(var_rows)
vol_states = pd.DataFrame(vol_state_rows)

return_diagnostics.to_csv(OUTPUT_DIR / "03_return_diagnostics_by_symbol.csv", index=False, encoding="utf-8-sig")
best_residual_diagnostics.to_csv(OUTPUT_DIR / "04_best_model_residual_diagnostics.csv", index=False, encoding="utf-8-sig")
var_backtests.to_csv(OUTPUT_DIR / "05_var_backtest_by_symbol.csv", index=False, encoding="utf-8-sig")
vol_states.to_csv(OUTPUT_DIR / "06_current_vol_state_and_position_scaling.csv", index=False, encoding="utf-8-sig")

print("传统模型检验完成。")
print("收益率诊断:", OUTPUT_DIR / "03_return_diagnostics_by_symbol.csv")
print("残差诊断:", OUTPUT_DIR / "04_best_model_residual_diagnostics.csv")
print("VaR 回测:", OUTPUT_DIR / "05_var_backtest_by_symbol.csv")
print("波动率状态与仓位缩放:", OUTPUT_DIR / "06_current_vol_state_and_position_scaling.csv")




二、传统模型检验
传统模型检验完成。
收益率诊断: 20_contract_vol_model_output/03_return_diagnostics_by_symbol.csv
残差诊断: 20_contract_vol_model_output/04_best_model_residual_diagnostics.csv
VaR 回测: 20_contract_vol_model_output/05_var_backtest_by_symbol.csv
波动率状态与仓位缩放: 20_contract_vol_model_output/06_current_vol_state_and_position_scaling.csv


In [13]:
# 三、论文模型检验：Hong-Lee 广义谱二阶导数检验
# ============================================================

print("\n" + "=" * 80)
print("三、论文模型检验：Hong-Lee 广义谱二阶导数检验")
print("=" * 80)

hl_candidate_cols = [
    c for c in candidate_results.columns
    if c.startswith("hl_") or c in [
        "symbol", "model_name", "model_usable", "traditional_diag_pass",
        "aic", "bic", "param_check_reason", "hl_moment_condition_ok",
        "hl_moment_condition_reason", "persistence_warning_flag",
        "persistence_warning_reason", "fit_warning_flag", "fit_warnings",
    ]
]
hl_all_candidates = candidate_results[hl_candidate_cols].copy()

hl_core_rows = []
hl_grid_rows = []
hl_sensitivity_rows = []
hl_dd_bandwidth_rows = []
hl_dd_selected_rows = []
model_health_rows = []

for symbol in HIGH_RELIABILITY_SYMBOLS:
    best_row = best_models[best_models["symbol"] == symbol]
    if best_row.empty:
        continue

    best_model_name = best_row["best_model_name"].iloc[0]
    fit = fit_store.get((symbol, best_model_name), None)

    if fit is None:
        continue

    z = pd.Series(fit["std_resid"]).replace([np.inf, -np.inf], np.nan).dropna()
    n = len(z)
    p0 = default_hl_bandwidth(n)

    core = hong_lee_kernel_test(z, p0)
    core.update({
        "symbol": symbol,
        "best_model_name": best_model_name,
        "sample_size": n,
        "base_bandwidth_p": p0,
    })
    hl_core_rows.append(core)

    symbol_grid_rows = []
    for mult in HL_BANDWIDTH_MULTIPLIERS:
        p = p0 * mult
        r = hong_lee_kernel_test(z, p)
        r.update({
            "symbol": symbol,
            "best_model_name": best_model_name,
            "sample_size": n,
            "bandwidth_multiplier": mult,
            "base_bandwidth_p": p0,
        })
        hl_grid_rows.append(r)
        symbol_grid_rows.append(r)

    sensitivity_df = run_hl_sensitivity_grid(
        z,
        p0=p0,
        label=f"{symbol}:{best_model_name}",
    )
    sensitivity_df["symbol"] = symbol
    sensitivity_df["best_model_name"] = best_model_name
    hl_sensitivity_rows.extend(sensitivity_df.to_dict("records"))

    if RUN_HL_DATA_DRIVEN_BANDWIDTH:
        dd_results, dd_selected = run_hl_data_driven_bandwidth(
            z,
            label=f"{symbol}:{best_model_name}",
        )
        dd_results["symbol"] = symbol
        dd_results["best_model_name"] = best_model_name
        dd_selected["symbol"] = symbol
        dd_selected["best_model_name"] = best_model_name
        hl_dd_bandwidth_rows.extend(dd_results.to_dict("records"))
        hl_dd_selected_rows.append(dd_selected)

    symbol_grid_df = pd.DataFrame(symbol_grid_rows)
    stability = summarize_hl_decisions(symbol_grid_df)

    core_decision = core.get("decision")
    traditional_pass = bool(best_row["traditional_diag_pass"].iloc[0])
    fit_warning_value = best_row.get("fit_warning_flag", pd.Series([False])).iloc[0]
    fit_warning_flag = bool(fit_warning_value) if pd.notna(fit_warning_value) else False
    moment_value = best_row.get("hl_moment_condition_ok", pd.Series([True])).iloc[0]
    hl_moment_condition_ok = bool(moment_value) if pd.notna(moment_value) else True
    persistence_warning_value = best_row.get("persistence_warning_flag", pd.Series([False])).iloc[0]
    persistence_warning_flag = bool(persistence_warning_value) if pd.notna(persistence_warning_value) else False

    health_raw = classify_model_health_from_hl(
        n=n,
        traditional_pass=traditional_pass,
        fit_warning_flag=fit_warning_flag,
        hl_moment_condition_ok=hl_moment_condition_ok,
        persistence_warning_flag=persistence_warning_flag,
        hl_decision=core_decision,
        bandwidth_stability=stability["bandwidth_stability"],
    )

    model_health_rows.append({
        "symbol": symbol,
        "best_model_name": best_model_name,
        "sample_size": n,
        "traditional_diag_pass": traditional_pass,
        "fit_warning_flag": fit_warning_flag,
        "hl_moment_condition_ok": hl_moment_condition_ok,
        "hl_moment_condition_reason": best_row.get("hl_moment_condition_reason", pd.Series(["ok"])).iloc[0],
        "persistence_warning_flag": persistence_warning_flag,
        "persistence_warning_reason": best_row.get("persistence_warning_reason", pd.Series(["ok"])).iloc[0],
        "hl_core_decision": core_decision,
        "hl_core_pvalue": core.get("pvalue"),
        "hl_core_M_robust": core.get("M_robust"),
        "hl_core_M_iid": core.get("M_iid"),
        "hl_core_Q_gsd": core.get("Q_gsd"),
        "hl_core_TQ_gsd": core.get("TQ_gsd"),
        **stability,
        "model_health_raw": health_raw,
        "model_health": health_raw,
    })

hl_core = pd.DataFrame(hl_core_rows)
hl_grid = pd.DataFrame(hl_grid_rows)
hl_sensitivity = pd.DataFrame(hl_sensitivity_rows)
hl_dd_bandwidth = pd.DataFrame(hl_dd_bandwidth_rows)
hl_dd_selected = pd.DataFrame(hl_dd_selected_rows)
model_health = pd.DataFrame(model_health_rows)

if not hl_core.empty:
    hl_core["pvalue_adj_across_symbols"] = adjust_pvalues(hl_core["pvalue"])
    hl_core["decision_adj_across_symbols"] = np.where(
        hl_core["pvalue_adj_across_symbols"].notna() & (hl_core["pvalue_adj_across_symbols"] < ALPHA),
        "REJECT",
        np.where(hl_core["pvalue_adj_across_symbols"].notna(), "FAIL_TO_REJECT", hl_core["decision"]),
    )

if not model_health.empty and not hl_core.empty:
    model_health = model_health.merge(
        hl_core[["symbol", "best_model_name", "pvalue_adj_across_symbols", "decision_adj_across_symbols"]],
        on=["symbol", "best_model_name"],
        how="left",
    )
    model_health["model_health_adjusted"] = model_health.apply(
        lambda r: classify_model_health_from_hl(
            n=int(r["sample_size"]) if pd.notna(r["sample_size"]) else 0,
            traditional_pass=bool(r["traditional_diag_pass"]),
            fit_warning_flag=bool(r["fit_warning_flag"]) if pd.notna(r["fit_warning_flag"]) else False,
            hl_moment_condition_ok=bool(r["hl_moment_condition_ok"]) if pd.notna(r.get("hl_moment_condition_ok", True)) else True,
            persistence_warning_flag=bool(r["persistence_warning_flag"]) if pd.notna(r.get("persistence_warning_flag", False)) else False,
            hl_decision=r["decision_adj_across_symbols"],
            bandwidth_stability=r["bandwidth_stability"],
        ),
        axis=1,
    )

if not model_health.empty and not hl_dd_selected.empty:
    model_health = model_health.merge(
        hl_dd_selected[
            [
                "symbol", "best_model_name", "dd_status", "dd_selected_p",
                "dd_selected_M_robust", "dd_selected_pvalue",
                "dd_selected_decision", "dd_post_selection_boundary",
            ]
        ],
        on=["symbol", "best_model_name"],
        how="left",
    )

mc_detail_rows = []
mc_summary_rows = []
if RUN_HL_MONTE_CARLO_SIZE:
    for family, spec_name in HL_MC_FAMILY_SPECS.items():
        spec = next((s for s in MODEL_SPECS if s.name == spec_name), None)
        if spec is None:
            mc_summary_rows.append({
                "family": family,
                "spec_name": spec_name,
                "status": "missing_spec",
            })
            continue

        usable = candidate_results[
            (candidate_results["model_name"] == spec_name) &
            (candidate_results["model_usable"])
        ].copy()

        if usable.empty:
            mc_summary_rows.append({
                "family": family,
                "spec_name": spec_name,
                "status": "no_usable_fit",
            })
            continue

        representative = usable.sort_values("sample_size", ascending=False).iloc[0]
        symbol = representative["symbol"]
        fit = fit_store.get((symbol, spec_name), None)
        if fit is None:
            mc_summary_rows.append({
                "family": family,
                "spec_name": spec_name,
                "symbol": symbol,
                "status": "missing_fit_object",
            })
            continue

        detail, summary_row = run_hl_monte_carlo_size_for_family(
            symbol=symbol,
            family=family,
            spec=spec,
            fit=fit,
            reps=HL_MC_REPS,
        )
        detail["mc_family_status"] = "representative_fit"
        mc_detail_rows.extend(detail.to_dict("records"))
        summary_row["status"] = "ok"
        mc_summary_rows.append(summary_row)

hl_mc_size_detail = pd.DataFrame(mc_detail_rows)
hl_mc_size_summary = pd.DataFrame(mc_summary_rows)

explanation_rows = []
for _, best in best_models.iterrows():
    symbol = best["symbol"]
    model_name = best["best_model_name"]
    mh = model_health[
        (model_health["symbol"] == symbol) &
        (model_health["best_model_name"] == model_name)
    ]
    mh_row = mh.iloc[0].to_dict() if not mh.empty else {}
    cand = candidate_results[candidate_results["symbol"] == symbol].copy()
    if "hl_decision" in cand and cand["hl_decision"].isin(["REJECT", "FAIL_TO_REJECT"]).any():
        rejected_raw = int((cand["hl_decision"] == "REJECT").sum())
    else:
        rejected_raw = np.nan
    usable_count = int(cand["model_usable"].sum()) if "model_usable" in cand else 0
    accepted_count = int(
        (
            (cand["model_usable"]) &
            (cand["traditional_diag_pass"])
        ).sum()
    ) if {"model_usable", "traditional_diag_pass"}.issubset(cand.columns) else 0

    reason_bits = [
        f"selected_status={best.get('best_status')}",
        f"criterion={best.get('selection_criterion')}",
        f"best_bic={best.get('best_bic')}",
        f"traditional_pass={best.get('traditional_diag_pass')}",
        f"hl_core_decision={mh_row.get('hl_core_decision')} p={mh_row.get('hl_core_pvalue')}",
        f"hl_adj_decision={mh_row.get('decision_adj_across_symbols')} adj_p={mh_row.get('pvalue_adj_across_symbols')}",
        f"model_health={mh_row.get('model_health')} model_health_adjusted={mh_row.get('model_health_adjusted')}",
    ]
    explanation_rows.append({
        "symbol": symbol,
        "best_model_name": model_name,
        "usable_candidate_count": usable_count,
        "accepted_candidate_count_traditional": accepted_count,
        "raw_hl_rejected_candidate_count": rejected_raw,
        "selection_explanation": " | ".join(str(x) for x in reason_bits),
    })

model_selection_explanations = pd.DataFrame(explanation_rows)

hl_all_candidates.to_csv(OUTPUT_DIR / "07_hong_lee_all_candidates_by_symbol.csv", index=False, encoding="utf-8-sig")
hl_core.to_csv(OUTPUT_DIR / "08_hong_lee_core_best_by_symbol.csv", index=False, encoding="utf-8-sig")
hl_grid.to_csv(OUTPUT_DIR / "09_hong_lee_bandwidth_grid_best_by_symbol.csv", index=False, encoding="utf-8-sig")
model_health.to_csv(OUTPUT_DIR / "10_model_health_summary.csv", index=False, encoding="utf-8-sig")
hl_sensitivity.to_csv(OUTPUT_DIR / "11_hong_lee_sensitivity_best_by_symbol.csv", index=False, encoding="utf-8-sig")
hl_dd_bandwidth.to_csv(OUTPUT_DIR / "12_hong_lee_data_driven_bandwidth_grid.csv", index=False, encoding="utf-8-sig")
hl_dd_selected.to_csv(OUTPUT_DIR / "13_hong_lee_data_driven_bandwidth_selected.csv", index=False, encoding="utf-8-sig")
hl_mc_size_detail.to_csv(OUTPUT_DIR / "14_hong_lee_monte_carlo_size_detail.csv", index=False, encoding="utf-8-sig")
hl_mc_size_summary.to_csv(OUTPUT_DIR / "15_hong_lee_monte_carlo_size_summary.csv", index=False, encoding="utf-8-sig")
model_selection_explanations.to_csv(OUTPUT_DIR / "16_model_selection_explanations.csv", index=False, encoding="utf-8-sig")

print("论文模型检验完成。")
print("Hong-Lee 全候选模型结果:", OUTPUT_DIR / "07_hong_lee_all_candidates_by_symbol.csv")
print("Hong-Lee 最优模型主带宽结果:", OUTPUT_DIR / "08_hong_lee_core_best_by_symbol.csv")
print("Hong-Lee 最优模型多带宽结果:", OUTPUT_DIR / "09_hong_lee_bandwidth_grid_best_by_symbol.csv")
print("模型健康度汇总:", OUTPUT_DIR / "10_model_health_summary.csv")
print("Hong-Lee 敏感性报告:", OUTPUT_DIR / "11_hong_lee_sensitivity_best_by_symbol.csv")
print("Hong-Lee data-driven bandwidth 选择:", OUTPUT_DIR / "13_hong_lee_data_driven_bandwidth_selected.csv")
print("Hong-Lee Monte Carlo size 汇总:", OUTPUT_DIR / "15_hong_lee_monte_carlo_size_summary.csv")
print("模型选择解释层:", OUTPUT_DIR / "16_model_selection_explanations.csv")




三、论文模型检验：Hong-Lee 广义谱二阶导数检验
论文模型检验完成。
Hong-Lee 全候选模型结果: 20_contract_vol_model_output/07_hong_lee_all_candidates_by_symbol.csv
Hong-Lee 最优模型主带宽结果: 20_contract_vol_model_output/08_hong_lee_core_best_by_symbol.csv
Hong-Lee 最优模型多带宽结果: 20_contract_vol_model_output/09_hong_lee_bandwidth_grid_best_by_symbol.csv
模型健康度汇总: 20_contract_vol_model_output/10_model_health_summary.csv
Hong-Lee 敏感性报告: 20_contract_vol_model_output/11_hong_lee_sensitivity_best_by_symbol.csv
Hong-Lee data-driven bandwidth 选择: 20_contract_vol_model_output/13_hong_lee_data_driven_bandwidth_selected.csv
Hong-Lee Monte Carlo size 汇总: 20_contract_vol_model_output/15_hong_lee_monte_carlo_size_summary.csv
模型选择解释层: 20_contract_vol_model_output/16_model_selection_explanations.csv


In [ ]:
# 4. 汇总输出
# ============================================================

print("\n" + "=" * 80)
print("最终汇总")
print("=" * 80)

summary = (
    best_models
    .merge(model_health, on=["symbol", "best_model_name"], how="left", suffixes=("", "_health"))
    .merge(vol_states, on=["symbol", "best_model_name"], how="left", suffixes=("", "_vol"))
)

summary.to_csv(OUTPUT_DIR / "10_final_summary_by_symbol.csv", index=False, encoding="utf-8-sig")

# Excel 汇总
excel_path = OUTPUT_DIR / "20_contract_model_selection_full_results.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    candidate_results.to_excel(writer, sheet_name="01_all_candidates", index=False)
    best_models.to_excel(writer, sheet_name="02_best_models", index=False)
    return_diagnostics.to_excel(writer, sheet_name="03_return_diag", index=False)
    best_residual_diagnostics.to_excel(writer, sheet_name="04_residual_diag", index=False)
    var_backtests.to_excel(writer, sheet_name="05_var_backtest", index=False)
    vol_states.to_excel(writer, sheet_name="06_vol_state", index=False)
    hl_all_candidates.to_excel(writer, sheet_name="07_hl_all_candidates", index=False)
    hl_core.to_excel(writer, sheet_name="08_hl_core_best", index=False)
    hl_grid.to_excel(writer, sheet_name="09_hl_grid_best", index=False)
    model_health.to_excel(writer, sheet_name="10_model_health", index=False)
    hl_sensitivity.to_excel(writer, sheet_name="11_hl_sensitivity", index=False)
    hl_dd_selected.to_excel(writer, sheet_name="12_hl_dd_selected", index=False)
    hl_dd_bandwidth.to_excel(writer, sheet_name="13_hl_dd_grid", index=False)
    hl_mc_size_summary.to_excel(writer, sheet_name="14_hl_mc_size", index=False)
    hl_mc_size_detail.to_excel(writer, sheet_name="15_hl_mc_detail", index=False)
    model_selection_explanations.to_excel(writer, sheet_name="16_selection_notes", index=False)
    summary.to_excel(writer, sheet_name="10_final_summary", index=False)

elapsed = time.time() - start_time

print("总耗时秒数:", round(elapsed, 2))
print("最终 summary:", OUTPUT_DIR / "10_final_summary_by_symbol.csv")
print("Excel 汇总:", excel_path)

print("\nBest model distribution:")
print(best_models["best_model_name"].value_counts(dropna=False))

print("\nModel health distribution:")
print(model_health["model_health"].value_counts(dropna=False))

print("\nDone.")